# 01_prompt_llama

This prompt wwas designed to compare with Deepseek and Mistral prompt for reporting purposes.

This is not the prompt that was desined to have a higher recall in llama, it was derived from previous "Prompt 13 (Safety): Avg ~3.2 (High Recall)".

In [2]:
PROMPT_TEXT = r'''
def build_prompt(
    tasks_str: str,
    clean_titles: list[str],
    domain: str,
    job_ad_title: str,
    job_sector_category: str,
    full_ad_text: str,
) -> str:
    n_candidates = len(clean_titles)
    numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(clean_titles))

    full_excerpt = (full_ad_text or "").strip()[:400]
    full_block = f"\nFULL AD EXCERPT:\n{full_excerpt}\n" if full_excerpt else ""

    body = f"""
Return ONLY a valid JSON object with exactly one key "drop". No extra text.

TASK
Audit occupation matches. DROP candidates that are NOT functional matches for the job.

KEEP POLICY
There are {n_candidates} candidates (1-based).
- Default: KEEP 2 to 3 candidates.
- KEEP 1 ONLY if you are certain every other candidate is clearly wrong.
- If more than 3 are valid, drop the most generic ones to fit the 3-candidate cap.
- When in doubt, KEEP rather than DROP if functionally plausible.

JOB CONTEXT (SOURCE OF TRUTH)
Title: {job_ad_title}
Sector: {job_sector_category}
Domain: {domain}
Tasks: {tasks_str}
{full_block}

CANDIDATES (1-based)
{numbered}

OUTPUT
Return ONLY JSON: {{"drop":[...]}}
""".strip()

    return body

def build_retry_prompt(
    tasks_str: str,
    clean_titles: list[str],
    domain: str,
    job_ad_title: str,
    job_sector_category: str,
) -> str:
    numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(clean_titles))

    body = f"""
Return ONLY valid JSON: {{"drop":[...]}}. No extra text.

Title: {job_ad_title}
Sector: {job_sector_category}
Domain: {domain}
Tasks: {tasks_str}

Candidates (1-based):
{numbered}

Rules:
- Keep 2 to 3 by default.
- Keep 1 only if very sure all others are clearly wrong.
- Drop only clearly wrong functions.

Output ONLY JSON: {{"drop":[...]}}
""".strip()

    return body
'''.strip()


# run

bge e5 and gte

In [7]:
import subprocess, re
from pathlib import Path

SBATCH = Path("/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/sbatch/run_llama_drop_bge.sbatch")
assert SBATCH.exists(), f"Missing sbatch file: {SBATCH}"

out = subprocess.check_output(["sbatch", str(SBATCH)], text=True)
jobid = re.search(r"(\d+)", out).group(1)
print("JOBID:", jobid)
bge_jobid = jobid

import subprocess, re
from pathlib import Path

SBATCH = Path("/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/sbatch/run_llama_drop_e5.sbatch")
assert SBATCH.exists(), f"Missing sbatch file: {SBATCH}"

out = subprocess.check_output(["sbatch", str(SBATCH)], text=True)
jobid = re.search(r"(\d+)", out).group(1)
print("JOBID:", jobid)
e5_jobid = jobid

import subprocess, re
from pathlib import Path

SBATCH = Path("/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/sbatch/run_llama_drop_gte.sbatch")
assert SBATCH.exists(), f"Missing sbatch file: {SBATCH}"

out = subprocess.check_output(["sbatch", str(SBATCH)], text=True)
jobid = re.search(r"(\d+)", out).group(1)
print("JOBID:", jobid)
gte_jobid = jobid

JOBID: 2266256
JOBID: 2266257
JOBID: 2266258


# monitor run

In [14]:
import time, subprocess
from pathlib import Path

LOG_DIR = Path("/projects/a5u/adu_dev/aisi-economy-index/logs")
def sh(cmd):
    return subprocess.run(cmd, shell=True, text=True, capture_output=True).stdout.strip()
while True:
    print("\n=== squeue ===")
    sq = sh(f"squeue -j {jobid}")
    print(sq if sq else "<finished>")

    out_logs = sorted(LOG_DIR.glob(f"*{jobid}*.out"))
    err_logs = sorted(LOG_DIR.glob(f"*{jobid}*.err"))

    if out_logs:
        print("\n--- STDOUT ---")
        print(sh(f"tail -n 20 {out_logs[-1]}"))

    if err_logs:
        print("\n--- STDERR ---")
        print(sh(f"tail -n 20 {err_logs[-1]}"))

    if not sq:
        break

    time.sleep(5)



=== squeue ===
<finished>

--- STDOUT ---
NPZ_DIR=/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection/gte_large
total 7,1M
drwxr-xr-x 2 autonomyluiz.a5u autonomyluiz.a5u 4,0K fev 10 21:18 adzuna_month01
-rw-r--r-- 1 autonomyluiz.a5u autonomyluiz.a5u 569K fev  5 16:02 adzuna_month01.npz
drwxr-xr-x 2 autonomyluiz.a5u autonomyluiz.a5u 4,0K fev 10 21:18 adzuna_month02
-rw-r--r-- 1 autonomyluiz.a5u autonomyluiz.a5u 570K fev  5 16:03 adzuna_month02.npz
drwxr-xr-x 2 autonomyluiz.a5u autonomyluiz.a5u 4,0K fev 10 21:18 adzuna_month03
-rw-r--r-- 1 autonomyluiz.a5u autonomyluiz.a5u 571K fev  5 16:03 adzuna_month03.npz
drwxr-xr-x 2 autonomyluiz.a5u autonomyluiz.a5u 4,0K fev 10 21:18 adzuna_month04
-rw-r--r-- 1 autonomyluiz.a5u autonomyluiz.a5u 571K fev  5 16:04 adzuna_month04.npz
drwxr-xr-x 2 autonomyluiz.a5u autonomyluiz.a5u 4,0K fev 10 21:18 adzuna_month05
EMBED=gte_large
[TASK] 9 -> adzuna_month10
[NPZ]  /projects/a5u/adu_dev/aisi-econ

# Report

In [20]:
import os
os.getcwd()
import importlib
import gen_report_helper as grh # gen_report_helper.py 
from pathlib import Path
importlib.reload(grh) 

res = grh.generate_LLM_MODEL_full_report_and_plots(
    JSONL_BASE_DIR=Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection/"),
    NPZ_BASE_DIR=Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection"),
    llm_model="meta-llama--Meta-Llama-3.1-8B-Instruct",
    prompt_text=PROMPT_TEXT,
    reports_subdir="reports/llama_reports")
    

res



Scanning Month 1 to 14 (Flexible format) for embeds: ('bge_large', 'e5_large', 'gte_large')
[SELECT] bge_large | adzuna_month01 -> llama_drop_only_adzuna_month01_0_1000_job2266259_task0_20260210_213823.jsonl
[SELECT] bge_large | adzuna_month02 -> llama_drop_only_adzuna_month02_0_1000_job2266260_task1_20260210_213823.jsonl
[SELECT] bge_large | adzuna_month03 -> llama_drop_only_adzuna_month03_0_1000_job2266261_task2_20260210_213820.jsonl
[SELECT] bge_large | adzuna_month04 -> llama_drop_only_adzuna_month04_0_1000_job2266262_task3_20260210_213820.jsonl
[SELECT] bge_large | adzuna_month05 -> llama_drop_only_adzuna_month05_0_1000_job2266263_task4_20260210_213818.jsonl
[SELECT] bge_large | adzuna_month06 -> llama_drop_only_adzuna_month06_0_1000_job2266264_task5_20260210_213819.jsonl
[SELECT] bge_large | adzuna_month07 -> llama_drop_only_adzuna_month07_0_1000_job2266265_task6_20260210_213829.jsonl
[SELECT] bge_large | adzuna_month08 -> llama_drop_only_adzuna_month08_0_1000_job2266266_task7_2

{'out_dir': '/lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/reports/llama_reports/20260210_221849',
 'txt_path': '/lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/reports/llama_reports/20260210_221849/20260210_221849_FULL_TEXT_REPORT.txt',
 'heatmap_path': '/lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/reports/llama_reports/20260210_221849/20260210_221849_avg_jaccard_heatmap.png'}

In [ ]:
#!/usr/bin/env python3
"""
Validate embedding file paths and check normalization status.
"""
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List

def check_norms(path: str, key: Optional[str] = None) -> Dict:
    """
    Check if embeddings exist and their normalization status.
    
    Args:
        path: Path to .npy or .npz file
        key: Key name if npz file (e.g., 'embeddings', 'role_embeds')
    
    Returns:
        Dict with validation results
    """
    p = Path(path)
    result = {
        "path": path,
        "exists": False,
        "normalized": None,
        "shape": None,
        "dtype": None,
        "mean_norm": None,
        "error": None
    }
    
    try:
        if not p.exists():
            result["error"] = "File not found"
            return result
        
        result["exists"] = True
        
        # Load embeddings
        if path.endswith(".npy"):
            emb = np.load(path)
        elif path.endswith(".npz"):
            if key is None:
                result["error"] = "NPZ file requires 'key' argument"
                return result
            data = np.load(path, allow_pickle=True)
            if key not in data.files:
                result["error"] = f"Key '{key}' not in NPZ. Available: {data.files}"
                return result
            emb = data[key]
        else:
            result["error"] = "Unknown file type (not .npy or .npz)"
            return result
        
        result["shape"] = emb.shape
        result["dtype"] = str(emb.dtype)
        
        # Check normalization (sample first 1000 rows for speed)
        sample_size = min(1000, len(emb))
        sample = emb[:sample_size].astype(np.float32)
        norms = np.linalg.norm(sample, axis=1)
        mean_norm = float(np.mean(norms))
        std_norm = float(np.std(norms))
        
        result["mean_norm"] = mean_norm
        result["std_norm"] = std_norm
        
        # Normalized if mean ~1.0 and low std
        if 0.99 <= mean_norm <= 1.01 and std_norm < 0.05:
            result["normalized"] = True
        else:
            result["normalized"] = False
        
    except Exception as e:
        result["error"] = str(e)
    
    return result


def main():
    """Validate all embedding paths."""
    
    # Define paths to check
    paths_to_check = [
        # O*NET embeddings
        {
            "type": "O*NET BGE",
            "path": "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_1/onet_embeddings/bge_large/onet_bge_large_embeddings.npy",
            "key": None
        },
        {
            "type": "O*NET E5",
            "path": "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_1/onet_embeddings/e5_large/onet_e5_large_embeddings.npy",
            "key": None
        },
        {
            "type": "O*NET GTE",
            "path": "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_1/onet_embeddings/gte_large/onet_gte_large_embeddings.npy",
            "key": None
        },
        # Target embeddings (jobs)
        {
            "type": "Target BGE",
            "path": "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2/dev/target_embeddings/bge_large/adzuna_month01/target_bge_large_month01_shard00.npz",
            "key": "embeddings"
        },
        {
            "type": "Target E5",
            "path": "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2/dev/target_embeddings/e5_large/adzuna_month01/target_e5_large_month01_shard00.npz",
            "key": "embeddings"
        },
        {
            "type": "Target GTE",
            "path": "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2/dev/target_embeddings/gte_large/adzuna_month01/target_gte_large_month01_shard00.npz",
            "key": "embeddings"
        },
    ]
    
    print("="*100)
    print("EMBEDDING PATH VALIDATION REPORT")
    print("="*100)
    print()
    
    results: List[Dict] = []
    
    for item in paths_to_check:
        print(f"Checking: {item['type']}")
        print(f"Path: {item['path']}")
        
        result = check_norms(item["path"], key=item.get("key"))
        result["type"] = item["type"]
        results.append(result)
        
        if result["exists"]:
            print(f"  ✅ EXISTS")
            print(f"  Shape: {result['shape']}")
            print(f"  Dtype: {result['dtype']}")
            if result["normalized"] is not None:
                norm_status = "✅ NORMALIZED" if result["normalized"] else "⚠️  NOT NORMALIZED"
                print(f"  {norm_status} (mean norm: {result['mean_norm']:.4f})")
        else:
            print(f"  ❌ NOT FOUND")
            if result["error"]:
                print(f"  Error: {result['error']}")
        print()
    
    # Summary table
    print("="*100)
    print("SUMMARY TABLE")
    print("="*100)
    print(f"{'Type':<20} {'Exists':<10} {'Normalized':<15} {'Shape':<20} {'Mean Norm':<12}")
    print("-"*100)
    
    for r in results:
        exists = "✅ Yes" if r["exists"] else "❌ No"
        normalized = "✅ Yes" if r["normalized"] else ("⚠️  No" if r["normalized"] is False else "N/A")
        shape = str(r["shape"]) if r["shape"] else "N/A"
        mean_norm = f"{r['mean_norm']:.4f}" if r["mean_norm"] else "N/A"
        
        print(f"{r['type']:<20} {exists:<10} {normalized:<15} {shape:<20} {mean_norm:<12}")
    
    print()
    print("="*100)
    
    # Check for issues
    issues = [r for r in results if not r["exists"] or r["error"]]
    if issues:
        print("⚠️  ISSUES FOUND:")
        for r in issues:
            print(f"  - {r['type']}: {r['error'] or 'File not found'}")
    else:
        print("✅ ALL PATHS VALID")
    
    print("="*100)


if __name__ == "__main__":
    main()